# 02 - Hiperparámetros, Calibración y Gran Benchmark de Regresión

## Curso de Machine Learning: Modelos de Regresión

En este cuaderno aprenderemos a calibrar los 3 hiperparámetros fundamentales de SVR ($C$, $\epsilon$, $\gamma$) mediante `GridSearchCV`, y concluiremos con el **Gran Benchmark Comparativo de Modelos de Regresión** evaluando en un mismo marco experimental los 6 algoritmos estudiados a lo largo del curso.

---

### Contenidos
1. Interacción entre Hiperparámetros en SVR: $C$, $\epsilon$ y $\gamma$
2. Búsqueda por Rejilla (`GridSearchCV`) en SVR
3. Gran Benchmark Comparativo:
   - Regresión Lineal / Ridge
   - Árbol de Decisión CART
   - Random Forest Regressor
   - KNN Regressor
   - XGBoost Regressor
   - Support Vector Regressor (SVR RBF)
4. Análisis de Compensaciones (*Trade-offs*): Precisión vs. Velocidad vs. Interpretabilidad
5. Conclusiones Generales del Módulo de Regresión


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVR
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. Carga de Datos y Optimización de SVR

In [ ]:
df = pd.read_csv("../../data/auto_mpg.csv")
features = ['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year']
X = df[features]
y = df['mpg']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

pipe_svr_tune = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR())
])

param_grid_svr = {
    'svr__C': [1.0, 10.0, 50.0, 100.0],
    'svr__epsilon': [0.1, 0.5, 1.0, 2.0],
    'svr__gamma': ['scale', 'auto', 0.01, 0.1]
}

grid_svr = GridSearchCV(pipe_svr_tune, param_grid_svr, cv=5, scoring='r2', n_jobs=-1)
grid_svr.fit(X_train, y_train)

print(f"Mejores Hiperparametros SVR: {grid_svr.best_params_}")
print(f"Mejor R2 CV:                 {grid_svr.best_score_:.4f}")

best_svr = grid_svr.best_estimator_
y_pred_svr = best_svr.predict(X_test)
print(f"R2 SVR Optimizado en Test:   {r2_score(y_test, y_pred_svr):.4f}")
print(f"RMSE en Test:                {np.sqrt(mean_squared_error(y_test, y_pred_svr)):.2f} mpg")


## 2. Gran Benchmark Comparativo de Modelos de Regresión

Evaluamos bajo condiciones estrictamente idénticas (mismo conjunto de prueba y preprocesamiento):
1. **Ridge (Regresión Regularizada L2)**
2. **Árbol de Decisión CART** (`max_depth=4`)
3. **KNN Regresor** ($K=7$ con `StandardScaler`)
4. **Random Forest Regresor** ($n=120$)
5. **XGBoost Regresor** ($n=120$, $\eta=0.08$)
6. **SVR (RBF)** (Optimizado con `StandardScaler`)


In [ ]:
modelos = {
    'Ridge (L2)': Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=1.0, random_state=42))]),
    'Arbol CART': DecisionTreeRegressor(max_depth=4, random_state=42),
    'KNN Regresor': Pipeline([('scaler', StandardScaler()), ('model', KNeighborsRegressor(n_neighbors=7, weights='distance'))]),
    'Random Forest': RandomForestRegressor(n_estimators=120, max_features='sqrt', random_state=42, n_jobs=-1),
    'XGBoost': xgb.XGBRegressor(n_estimators=120, learning_rate=0.08, max_depth=3, random_state=42),
    'SVR (RBF)': best_svr
}

bench_results = []

for nombre, mod in modelos.items():
    t0 = time.perf_counter()
    mod.fit(X_train, y_train)
    t_train = (time.perf_counter() - t0) * 1000 # ms
    
    t0 = time.perf_counter()
    preds = mod.predict(X_test)
    t_pred = (time.perf_counter() - t0) * 1000 # ms
    
    r2 = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    
    bench_results.append({
        'Modelo': nombre,
        'R² Test': r2,
        'RMSE (mpg)': rmse,
        'MAE (mpg)': mae,
        'Train Time (ms)': t_train,
        'Pred Time (ms)': t_pred
    })

df_bench = pd.DataFrame(bench_results).sort_values(by='R² Test', ascending=False)
print("Resultados del Gran Benchmark de Regresion:
")
print(df_bench.to_string(index=False))


In [ ]:
# Visualizacion Grafica del Benchmark
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

df_bench.plot(kind='barh', x='Modelo', y='R² Test', ax=axes[0], color='#2b5c8f', legend=False)
axes[0].set_title('Coeficiente de Determinacion R² en Test', fontweight='bold')
axes[0].set_xlim(0.70, 0.95)
axes[0].set_xlabel('R² (Mayor es mejor)')
axes[0].invert_yaxis()

df_bench.plot(kind='barh', x='Modelo', y='RMSE (mpg)', ax=axes[1], color='#d95f02', legend=False)
axes[1].set_title('Error RMSE en Test (Millas por Galon)', fontweight='bold')
axes[1].set_xlabel('RMSE (Menor es mejor)')
axes[1].invert_yaxis()

plt.suptitle('Comparativa Global de Algoritmos de Regresion en Auto MPG', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 3. Resumen y Guía de Selección para Producción

| Algoritmo | Ventajas Principales | Desventajas | Caso de Uso Recomendado |
|---|---|---|---|
| **Regresión Lineal / Ridge** | Máxima interpretabilidad, inferencia ultrarrápida ($O(D)$) | No captura no linealidades sin transformación manual | Relaciones netamente lineales, baselines iniciales |
| **KNN Regresor** | No paramétrico, sin supuestos de distribución | Inferencia lenta ($O(ND)$), no extrapola | Interpolación local en baja dimensión ($D < 15$) |
| **Árbol de Decisión** | Reglas transparentes, maneja datos tabulares mixtos | Alta varianza, saltos escalonados discontinuos | Explicabilidad regulatoria estricta |
| **Random Forest** | Excelente precisión, robusto ante ruido, no sobreajusta por $n$ | Modelo caja negra voluminoso en memoria | Estándar de oro para datos tabulares |
| **XGBoost** | Precisión de punta en competencias, velocidad extrema | Requiere sintonización cuidadosa para no sobreajustar | Grandes volúmenes de datos, máxima precisión |
| **SVR (RBF)** | Margen robusto ante outliers, alta elegancia matemática | Entrenamiento $O(N^2)$ a $O(N^3)$, requiere escalado estricto | Datasets pequeños a medianos con alta no linealidad |
